<a href="https://colab.research.google.com/github/Paridhi1312/Internship-BIts/blob/main/Github(4th_link).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q transformers peft accelerate bitsandbytes trl datasets scipy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.0 MB/s eta 0:00:00


In [4]:
import re
import random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import Dataset
from scipy.optimize import minimize
from scipy.stats import skew, kurtosis

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

tqdm.pandas()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ==========================================================
# Step 2: Mount Drive so nothing vanishes on disconnect
# ==========================================================
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = "/content/drive/MyDrive/sch10_project"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

MODEL_NAME = "NousResearch/Llama-2-7b-chat-hf"   # ungated mirror of Meta's Llama-2-7b-chat
MAX_SEQ_LEN = 512


Using device: cuda
Mounted at /content/drive


In [7]:
acd = pd.read_csv('/content/acd_concepts_clean.csv')

CONSTRAINT_NAME_MAP = {
    "standard_deviation": "sd",
    "std_dev": "sd",
    "90th_percentile": "90th_percentile",
    "percentile_90": "90th_percentile",
}

def parse_prompt(prompt_text):
    n_match = re.search(r"Generate\s+(\d+)\s+observations", prompt_text)
    n = int(n_match.group(1)) if n_match else None
    seg_match = re.search(r"satisfying:\s*(.*?)\.\s*Return", prompt_text)
    constraints = []
    if seg_match:
        for part in seg_match.group(1).split(","):
            m = re.search(r"([A-Za-z0-9 ]+?)\s*=\s*(-?\d+\.?\d*)", part.strip())
            if m:
                name = m.group(1).strip().lower().replace(" ", "_")
                name = CONSTRAINT_NAME_MAP.get(name, name)
                constraints.append((name, float(m.group(2))))
    return constraints, n

acd[["constraints", "n"]] = acd["original_prompt"].apply(lambda p: pd.Series(parse_prompt(p)))
print(f"Parsed {len(acd)} prompts.")

# ==========================================================
# Step 4: Solver -> builds the supervised training target
# ==========================================================
def stat_fn(name, arr):
    if name == "mean": return arr.mean()
    if name == "median": return np.median(arr)
    if name == "mode":
        vals, counts = np.unique(np.round(arr, 1), return_counts=True)
        return vals[np.argmax(counts)]
    if name == "variance": return arr.var()
    if name == "sd": return arr.std()
    if name == "skewness": return skew(arr)
    if name == "kurtosis": return kurtosis(arr)
    if name == "90th_percentile": return np.percentile(arr, 90)
    raise ValueError(f"Unsupported constraint: {name}")

def solve_joint(constraints, n, seed=0):
    non_corr = [(k, v) for k, v in constraints if k != "correlation"]
    corr = next((v for k, v in constraints if k == "correlation"), None)
    rng = np.random.default_rng(seed)
    x0 = rng.normal(loc=50, scale=10, size=n)

    def loss(x):
        total = 0.0
        for name, target in non_corr:
            try:
                total += (stat_fn(name, x) - target) ** 2
            except Exception:
                total += 1e6
        return total

    if non_corr:
        result = minimize(loss, x0, method="L-BFGS-B", options={"maxiter": 200, "maxfun": 2000})
        x = result.x
    else:
        x = x0

    if corr is not None:
        y_noise = rng.normal(0, 1, size=n)
        y = corr * (x - x.mean()) / (x.std() + 1e-8) + np.sqrt(max(1 - corr**2, 0)) * y_noise
        y = y * x.std() + x.mean()
        return np.round(x, 2).tolist(), np.round(y, 2).tolist()
    return np.round(x, 2).tolist(), None

def build_target_text(row):
    constraints, n = row["constraints"], row["n"]
    if not constraints or n is None:
        return None
    x, y = solve_joint(constraints, n)
    if y is not None:
        return ", ".join(f"({a}, {b})" for a, b in zip(x, y))
    return ", ".join(str(v) for v in x)

print("Solving constraints for each row...")
acd["target_observations"] = acd.progress_apply(build_target_text, axis=1)
acd = acd.dropna(subset=["target_observations"]).reset_index(drop=True)
print(f"Built {len(acd)} usable training examples.")

def build_input_text(row):
    return (
        f"Concept: {row['concept']}\n\nWorked Example: {row['worked_example']}\n\n"
        f"Task: {row['original_prompt']}\nObservations:"
    )

acd["input_text"] = acd.apply(build_input_text, axis=1)

# ==========================================================
# Step 5: Load Llama-2-7b-chat in 4-bit (QLoRA), mlabonne's config
# ==========================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

print("Loading base model (this can take a few minutes on first download)...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Model loaded.")

Parsed 225 prompts.
Solving constraints for each row...


100%|██████████| 225/225 [04:45<00:00,  1.27s/it]


Built 225 usable training examples.
Loading base model (this can take a few minutes on first download)...


config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

Model loaded.


In [8]:
def format_llama_prompt(input_text):
    return f"<s>[INST] {input_text} [/INST]"

def generate_observations(model, row, max_new_tokens=250):
    prompt = format_llama_prompt(row["input_text"])
    ids = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(
        **ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    return decoded.split("[/INST]")[-1].strip()

In [9]:
random.seed(1)
sample_idx = random.sample(range(len(acd)), min(20, len(acd)))
sample_rows = acd.iloc[sample_idx].reset_index(drop=True)

print("Generating baseline outputs...")
baseline_results = []
for _, row in tqdm(sample_rows.iterrows(), total=len(sample_rows)):
    gen = generate_observations(base_model, row)
    baseline_results.append({
        "prompt": row["original_prompt"],
        "constraints": row["constraints"],
        "n": row["n"],
        "generated_baseline": gen,
    })

baseline_df = pd.DataFrame(baseline_results)
baseline_df.to_csv(f"{PROJECT_DIR}/baseline_llama_results.csv", index=False)
print("Saved baseline generations.")

Generating baseline outputs...


  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 20/20 [05:33<00:00, 16.65s/it]

Saved baseline generations.


In [10]:
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [11]:
acd["text"] = acd.apply(lambda r: format_llama_prompt(r["input_text"]) + " " + r["target_observations"] + " </s>", axis=1)

train_df = acd.sample(frac=0.85, random_state=42)
test_df = acd.drop(train_df.index)

train_ds = Dataset.from_pandas(train_df[["text"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[["text"]].reset_index(drop=True))


In [18]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=f"{PROJECT_DIR}/llama_qlora_ckpt",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    optim="paged_adamw_32bit",
    report_to="none",
    dataset_text_field="text",     # <-- moved here
    max_seq_length=MAX_SEQ_LEN,    # <-- moved here too
    packing=False,
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer,   # `tokenizer=` kwarg was also renamed in recent trl
)


TypeError: SFTConfig.__init__() got an unexpected keyword argument 'max_seq_length'

In [19]:
import trl
print(trl.__version__)

1.7.1


In [20]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=f"{PROJECT_DIR}/llama_qlora_ckpt",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    optim="paged_adamw_32bit",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,      # <-- correct name for your trl version
    packing=False,
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/191 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/191 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/191 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/34 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/34 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/34 [00:00<?, ? examples/s]

In [21]:
ft_model = trainer.model
ft_model.eval()

print("Generating fine-tuned outputs...")
ft_results = []
for _, row in tqdm(sample_rows.iterrows(), total=len(sample_rows)):
    gen = generate_observations(ft_model, row)
    ft_results.append({"prompt": row["original_prompt"], "generated_finetuned": gen})

ft_df = pd.DataFrame(ft_results)
ft_df.to_csv(f"{PROJECT_DIR}/finetuned_llama_results.csv", index=False)

comparison = baseline_df.merge(ft_df, on="prompt")
comparison.to_csv(f"{PROJECT_DIR}/llama_before_after_comparison.csv", index=False)


Generating fine-tuned outputs...


  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 20/20 [09:24<00:00, 28.24s/it]


In [23]:
# ==========================================================
# Step 12: VALIDATION — recompute actual statistics and compare
# ==========================================================
def extract_numbers(text):
    return [float(n) for n in re.findall(r"-?\d+\.?\d*", text)]

def extract_pairs(text):
    return [(float(a), float(b)) for a, b in re.findall(r"\(\s*(-?\d+\.?\d*)\s*,\s*(-?\d+\.?\d*)\s*\)", text)]

def validate_generation(constraints, generated_text, n_expected, tol_frac=0.05):
    records = []
    nums = extract_numbers(generated_text)
    pairs = extract_pairs(generated_text)
    for name, target in constraints:
        rec = {
            "constraint_name": name, "target_value": target,
            "achieved_value": None, "abs_error": None, "rel_error_pct": None,
            "n_expected": n_expected, "n_generated": len(nums),
            "passed": False, "error_note": None,
        }
        try:
            if name == "correlation":
                if len(pairs) < 2:
                    rec["error_note"] = "Fewer than 2 (x, y) pairs found."
                    records.append(rec); continue
                xs, ys = zip(*pairs)
                achieved = np.corrcoef(xs, ys)[0, 1]
            else:
                if not nums:
                    rec["error_note"] = "No numeric observations found."
                    records.append(rec); continue
                achieved = stat_fn(name, np.array(nums))

            abs_err = abs(achieved - target)
            rec["achieved_value"] = round(float(achieved), 4)
            rec["abs_error"] = round(abs_err, 4)
            rec["rel_error_pct"] = round(100 * abs_err / abs(target), 2) if target != 0 else None
            tol = max(tol_frac * abs(target), 0.5)
            rec["passed"] = abs_err <= tol
            rec["error_note"] = "Within tolerance." if rec["passed"] else (
                f"{name}={rec['achieved_value']} vs target={target} (off by {rec['abs_error']})"
            )
        except Exception as e:
            rec["error_note"] = f"Validation error: {e}"
        records.append(rec)
    return records

def validate_dataframe(df, generated_col, label):
    rows = []
    for _, row in df.iterrows():
        for rec in validate_generation(row["constraints"], row[generated_col], row["n"]):
            rec["prompt"] = row["prompt"]
            rec["model"] = label
            rows.append(rec)
    return pd.DataFrame(rows)

# comparison already has 'constraints' and 'n' from baseline_df (Step 7) — no re-merge needed
baseline_validation = validate_dataframe(comparison, "generated_baseline", "baseline")
finetuned_validation = validate_dataframe(comparison, "generated_finetuned", "finetuned")

validation_report = pd.concat([baseline_validation, finetuned_validation], ignore_index=True)
validation_report.to_csv(f"{PROJECT_DIR}/validation_report_llama.csv", index=False)

summary = (
    validation_report.groupby("model")
    .agg(pass_rate=("passed", "mean"),
         avg_abs_error=("abs_error", "mean"),
         avg_rel_error_pct=("rel_error_pct", "mean"))
    .reset_index()
)
summary.to_csv(f"{PROJECT_DIR}/validation_summary_llama.csv", index=False)

print("\n=== Validation summary (baseline vs fine-tuned) ===")
print(summary)

print("\n=== Sample failing cases (fine-tuned) ===")
print(validation_report[(validation_report["model"] == "finetuned") & (~validation_report["passed"])]
      [["prompt", "constraint_name", "target_value", "achieved_value", "error_note"]].head(10))


=== Validation summary (baseline vs fine-tuned) ===
       model  pass_rate  avg_abs_error  avg_rel_error_pct
0   baseline   0.102941     672.846567        1265.389833
1  finetuned   0.102941      93.072730         456.555500

=== Sample failing cases (fine-tuned) ===
                                               prompt constraint_name  \
69  Generate 100 observations satisfying: mode=29....     correlation   
70  Generate 10 observations satisfying: mean=77.6...            mean   
71  Generate 10 observations satisfying: mean=77.6...        variance   
72  Generate 10 observations satisfying: mean=77.6...          median   
73  Generate 10 observations satisfying: mean=77.6...     correlation   
75  Generate 25 observations satisfying: skewness=...        skewness   
77  Generate 25 observations satisfying: skewness=...        kurtosis   
78  Generate 25 observations satisfying: skewness=...     correlation   
79  Generate 10 observations satisfying: variance=...        variance   
